In [1]:
import re
import pandas as pd
import xlrd
import xlea
from xlea import Schema, Column, config
pd.options.display.float_format = '{:,.2f}'.format 

In [2]:
FILE = "Приложение №2_смета_ЛС-02-01-01.xls"
COLS = ["номер", "обоснование", "наименование", "ед_изм", "кол_на_ед", "коэф1",
        "всего_коэф", "база_ед", "индекс", "тек_ед", "коэф2", "всего_руб"]

def norm(v):
    t = str(v).strip()
    return t[:-2] if t.endswith(".0") else t

def money(v):
    if v is None or norm(v) == "": return None
    if isinstance(v, (int, float)): return float(v)
    try: return float(str(v).replace(",", "").replace(" ", ""))
    except ValueError: return None          # «2,2\n(1,1*2)» и т.п. -> NaN

def raw_rows():
    sh = xlrd.open_workbook(FILE).sheet_by_index(0)
    return [sh.row_values(i) for i in range(sh.nrows)]

In [3]:
@config(header_rows=2)   # блок [подзаголовки + строка номеров]
class Row(Schema):
    c1:  str = Column(r"(^|;)1(?:\.0)?$",  regexp=True)
    c2:  str = Column(r"(^|;)2(?:\.0)?$",  regexp=True)
    c3:  str = Column(r"(^|;)3(?:\.0)?$",  regexp=True)
    c4:  str = Column(r"(^|;)4(?:\.0)?$",  regexp=True)
    c5:  str = Column(r"(^|;)5(?:\.0)?$",  regexp=True)
    c6:  str = Column(r"(^|;)6(?:\.0)?$",  regexp=True)
    c7:  str = Column(r"(^|;)7(?:\.0)?$",  regexp=True)
    c8:  str = Column(r"(^|;)8(?:\.0)?$",  regexp=True)
    c9:  str = Column(r"(^|;)9(?:\.0)?$",  regexp=True)
    c10: str = Column(r"(^|;)10(?:\.0)?$", regexp=True)
    c11: str = Column(r"(^|;)11(?:\.0)?$", regexp=True)
    c12: str = Column(r"(^|;)12(?:\.0)?$", regexp=True)

def read_all():
    try:
        rows = list(xlea.autoread(FILE, schema=Row))
        df = pd.DataFrame([r.asdict() for r in rows])
        df["row_index"] = [r.row_index for r in rows]
        return df.rename(columns={f"c{i+1}": n for i, n in enumerate(COLS)})
    except Exception:                        # фолбэк: карта колонок по строке «1…12»
        R = raw_rows()
        ni = next(i for i, r in enumerate(R) if [norm(c) for c in r[:3]] == ["1", "2", "3"])
        pos = {int(norm(c)): j for j, c in enumerate(R[ni]) if norm(c).isdigit()}
        data = []
        for i, r in enumerate(R[ni+1:], ni+1):
            rec = {n: (r[j] if j is not None and j < len(r) else "") for n, j in
                   ((n, pos.get(k+1)) for k, n in enumerate(COLS))}
            rec["row_index"] = i
            data.append(rec)
        return pd.DataFrame(data)

df = read_all()
print(df.shape)

(2252, 13)


In [4]:
# убираем пустые строки и шапки страниц
df = df[~(df[COLS].apply(lambda r: all(norm(v) == "" for v in r), axis=1))].copy()
df = df[~df.apply(lambda r: any("Приложение №" in norm(v) for v in r[COLS]), axis=1)].copy()

# числа
for c in ["кол_на_ед", "коэф1", "всего_коэф", "база_ед", "индекс", "тек_ед", "коэф2", "всего_руб"]:
    df[c] = df[c].map(money)
for c in COLS[:4]:
    df[c] = df[c].map(norm)

# раздел / подраздел тянем из колонки «номер» (туда попадают заголовки)
is_sec  = df["номер"].str.startswith("Раздел", na=False)
is_sub = df["номер"].str.contains("работы|водоотведение|Водоснабжение|санузла|Строительный мусор", na=False) & ~is_sec
df["раздел"] = df["номер"].where(is_sec).ffill()
df["вид_работ"] = df["номер"].where(is_sub).ffill()

# тип работы: чел.-ч / маш.-ч -> услуги, остальное -> материалы
df["тип_работы"] = df["ед_изм"].map(lambda u: "услуги" if u in ("чел.-ч", "маш.-ч")
                                    else ("материалы" if u not in ("", "%", None) else ""))
# категория пока заглушка (вместо трёх)
df["категория"] = "монтажные работы"
df.head(20)

,номер,обоснование,наименование,ед_изм,кол_на_ед,коэф1,всего_коэф,база_ед,индекс,тек_ед,коэф2,всего_руб,row_index,раздел,вид_работ,тип_работы,категория
0,Раздел 1. Адаптация прилегающей территории,,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,Раздел 1. Адаптация прилегающей территории,NaN,,монтажные работы
1,Демонтажные работы,,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,Раздел 1. Адаптация прилегающей территории,Демонтажные работы,,монтажные работы
2,1,ГЭСН13-06-003-01,Очистка поверхности щетками (Снятие старой кра...,м2,3.63,1.00,3.63,NaN,NaN,NaN,NaN,NaN,2,Раздел 1. Адаптация прилегающей территории,Демонтажные работы,материалы,монтажные работы
3,,,"Объем=1,21*3",,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,Раздел 1. Адаптация прилегающей территории,Демонтажные работы,,монтажные работы
4,,1,ОТ(ЗТ),чел.-ч,NaN,NaN,3.27,NaN,NaN,NaN,NaN,"1,136.72",4,Раздел 1. Адаптация прилегающей территории,Демонтажные работы,услуги,монтажные работы
5,,1-100-30,"Средний разряд работы 3,0",чел.-ч,0.90,NaN,3.27,NaN,NaN,347.94,NaN,"1,136.72",5,Раздел 1. Адаптация прилегающей территории,Демонтажные работы,услуги,монтажные работы
6,,,Итого прямые затраты,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"1,136.72",6,Раздел 1. Адаптация прилегающей территории,Демонтажные работы,,монтажные работы
7,,,ФОТ,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"1,136.72",7,Раздел 1. Адаптация прилегающей территории,Демонтажные работы,,монтажные работы
8,,Пр/812-013.0-1,НР Защита строительных конструкций и оборудова...,%,94.00,NaN,94.00,NaN,NaN,NaN,NaN,"1,068.52",8,Раздел 1. Адаптация прилегающей территории,Демонтажные работы,,монтажные работы
9,,Пр/774-013,СП Защита строительных конструкций и оборудова...,%,51.00,NaN,51.00,NaN,NaN,NaN,NaN,579.73,9,Раздел 1. Адаптация прилегающей территории,Демонтажные работы,,монтажные работы


In [5]:
def scan(labels):
    out = {}
    for r in raw_rows():
        vals = [norm(v) for v in r]
        for i, v in enumerate(vals):
            if v in labels and v not in out:
                for x in r[i+1:]:
                    if isinstance(x, (int, float)):   # без "and x" — ноль тоже значение
                        out[v] = float(x); break
    return out

fin = scan(["Всего", "Всего ФОТ (справочно)", "Эксплуатация машин", "Оплата труда рабочих",
            "Оплата труда машинистов (Отм)", "Материалы", "ВСЕГО по смете", "Оборудование"])
hdr = scan(["Сметная стоимость", "строительных работ", "монтажных работ", "оборудования",
            "прочих затрат", "Средства на оплату труда рабочих", "Средства на оплату труда машинистов",
            "Нормативные затраты труда рабочих", "Нормативные затраты труда машинистов"])
pd.Series(fin), pd.Series(hdr)

(Оплата труда рабочих              236,954.26
 Эксплуатация машин                 29,009.40
 Оплата труда машинистов (Отм)      14,705.04
 Материалы                       5,005,866.86
 Оборудование                      906,910.89
 Всего                           6,630,645.55
 Всего ФОТ (справочно)             251,659.30
 ВСЕГО по смете                  8,059,801.88
 dtype: float64,
 Сметная стоимость                      8,059.80
 строительных работ                     5,449.64
 Средства на оплату труда рабочих         236.95
 монтажных работ                          274.09
 Средства на оплату труда машинистов       14.71
 оборудования                             906.91
 Нормативные затраты труда рабочих        665.50
 прочих затрат                              0.00
 Нормативные затраты труда машинистов      34.00
 dtype: float64)

In [6]:
# ---------- шапка сметы (все ключи, ноль тоже значение) ----------
def _num(v):
    if isinstance(v, (int, float)): return float(v)
    s = str(v).replace(",", "").replace(" ", "").strip()
    try: return float(s)
    except ValueError: return None

HDR_KEYS = ["Сметная стоимость", "строительных работ", "монтажных работ", "оборудования", "прочих затрат",
            "Средства на оплату труда рабочих", "Средства на оплату труда машинистов",
            "Нормативные затраты труда рабочих", "Нормативные затраты труда машинистов"]
hdr2 = {}
for r in raw_rows():
    vals = [str(c).strip() for c in r]
    for i, v in enumerate(vals):
        if v in HDR_KEYS and v not in hdr2:
            for x in r[i+1:]:
                f = _num(x)
                if f is not None:
                    hdr2[v] = f
                    break

# ---------- посчитано по строкам df ----------
is_total  = df["наименование"].str.strip() == "Всего по позиции"
equip_sum, is_equip = 0.0, False
for num, name, tot in df[["номер", "наименование", "всего_руб"]].itertuples(index=False):
    n = str(num).strip()
    if re.fullmatch(r"\d+(?:\.\d+)?\s*[ОO]", n): is_equip = True
    elif re.fullmatch(r"\d+(?:\.\d+)?", n):     is_equip = False
    if str(name).strip() == "Всего по позиции" and is_equip:
        equip_sum += float(tot or 0)

total_pos = df.loc[is_total, "всего_руб"].sum()
total_nds = ((total_pos - equip_sum) * 1.015 + equip_sum) * 1.2   # зимние 1,5% + НДС 20%

is_labor  = (df["ед_изм"] == "чел.-ч") & df["наименование"].str.startswith(("Средний разряд работы", "Рабочий"), na=False)
is_machop = df["наименование"].str.startswith("ОТм(Зтм) Средний разряд машинистов", na=False)
labor_sum,  machop_sum  = df.loc[is_labor, "всего_руб"].sum(),  df.loc[is_machop, "всего_руб"].sum()
labor_h,    machop_h    = df.loc[is_labor, "кол_на_ед"].sum(),  df.loc[is_machop, "кол_на_ед"].sum()

# ---------- сверка: 3 колонки, без assert ----------
check = pd.DataFrame({
    "в начале": [hdr2["Сметная стоимость"] * 1000,
                 (hdr2["строительных работ"] + hdr2["монтажных работ"]
                  + hdr2["оборудования"] + hdr2["прочих затрат"]) * 1000,
                 hdr2["оборудования"] * 1000,
                 hdr2["Средства на оплату труда рабочих"] * 1000,
                 hdr2["Средства на оплату труда машинистов"] * 1000,
                 hdr2["Нормативные затраты труда рабочих"],
                 hdr2["Нормативные затраты труда машинистов"]],
    "посчитано": [total_nds, total_pos, equip_sum, labor_sum, machop_sum, labor_h, machop_h],
}, index=["Сметная стоимость с НДС, руб.", "Всего по позициям, руб.", "в т. ч. оборудование, руб.",
          "Оплата труда рабочих, руб.", "Оплата труда машинистов, руб.",
          "Затраты труда рабочих, чел.-ч", "Затраты труда машинистов, чел.-ч"])

check["разница"] = (check["посчитано"] - check["в начале"]).round(2)
check

,в начале,посчитано,разница
"Сметная стоимость с НДС, руб.","8,059,800.00","8,059,801.88",1.88
"Всего по позициям, руб.","6,630,640.00","6,630,645.55",5.55
"в т. ч. оборудование, руб.","906,910.00","906,910.89",0.89
"Оплата труда рабочих, руб.","236,950.00","236,954.26",4.26
"Оплата труда машинистов, руб.","14,710.00","14,705.04",-4.96
"Затраты труда рабочих, чел.-ч",665.50,"2,877.45","2,211.95"
"Затраты труда машинистов, чел.-ч",34.00,159.21,125.21


In [8]:
df.to_csv("ЛС-02-01-01_все_строки.csv", index=False, encoding="utf-8-sig")
check.to_csv("ЛС-02-01-01_сверка.csv", index=False, encoding="utf-8-sig")
print("готово:", df.shape)

готово: (2237, 17)
